In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

In [597]:
def integrate_rk4(func, x0, dt, steps, **kwargs):
    x = np.zeros((steps, len(x0)))
    x[0] = x0
    for i in range(1, steps):
        k1 = func(x[i-1], **kwargs)
        k2 = func(x[i-1] + 0.5 * dt * k1, **kwargs)
        k3 = func(x[i-1] + 0.5 * dt * k2, **kwargs)
        k4 = func(x[i-1] + dt * k3, **kwargs)
        x[i] = x[i-1] + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
    return x

In [598]:
def func_lorentz(x, sigma, rho, beta):
    F = np.zeros_like(x)
    F[...,0] = sigma * (x[...,1] - x[...,0])
    F[...,1] = x[...,0] * (rho - x[...,2]) - x[...,1]
    F[...,2] = x[...,0] * x[...,1] - beta * x[...,2]
    return F
def func_lorentz_jacobian(x, sigma, rho, beta):
    if x.ndim == 1:
        J = np.zeros((3, 3))
    else:
        J = np.zeros(x.shape[:-1] + (3, 3))
    J[...,0, 0] = -sigma
    J[...,0, 1] = sigma
    J[...,1, 0] = rho - x[...,2]
    J[...,1, 1] = -1
    J[...,1, 2] = -x[...,0]
    J[...,2, 0] = x[...,1]
    J[...,2, 1] = x[...,0]
    J[...,2, 2] = -beta
    return J

In [599]:
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
u0 = np.random.rand(3) * 30 - 15
dt = 0.001
simulation_steps = 300000
start_time = 50000
u = integrate_rk4(func_lorentz, u0, dt, simulation_steps, sigma=sigma, rho=rho, beta=beta)
u = u[start_time:]
#u_moving = np.cumsum(u, axis=0) / np.arange(1, len(u) + 1)[:, None]
#u_whitened = (u - np.mean(u, axis=0)) / np.std(u, axis=0)
from sklearn.decomposition import PCA
u_whitened = PCA(whiten=True).fit_transform(u)

In [600]:
J = func_lorentz_jacobian(u, sigma, rho, beta)
G = func_lorentz(u, sigma, rho, beta)
detJ = np.linalg.det(J)
energy_grad = np.einsum('...ij,...i->...j', J, G)
rotation = J - np.swapaxes(J, -1, -2)
accel = np.einsum('...ij,...j->...i', rotation, G) + energy_grad

In [601]:
%matplotlib qt
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(u[:, 0], u[:, 1], u[:, 2], label='Original')
ax.legend()
plt.show()

In [ ]:
def esn_softmax(x0, W, W_in, u, beta, dt):
    T = u.shape[0]
    K = W.shape[0]
    N = W.shape[1]
    x = np.zeros((T, N))
    x[0] = x0
    for t in range(1, T):
        logits = beta*(W @ x[t-1])
        logits_max = np.max(logits)
        exp_logits = np.exp(logits - logits_max)
        softmax = exp_logits / np.sum(exp_logits)
        x[t] = x[t-1]  + dt * ( softmax  @ W  - x[t-1] + W_in @ u[t-1])
    return x

def esn_softmax2(x0, W_inner, W_outer, W_in, u, beta, dt):
    T = u.shape[0]
    K = W_inner.shape[0]
    N = W_inner.shape[1]
    x = np.zeros((T, N))
    x[0] = x0
    for t in range(1, T):
        logits = beta*(W_inner @ x[t-1])
        logits_max = np.max(logits)
        exp_logits = np.exp(logits - logits_max)
        softmax = exp_logits / np.sum(exp_logits)
        x[t] = x[t-1]  + dt * ( softmax  @ W_outer  - x[t-1] + W_in @ u[t-1])
    return x

def esn_tanh(x0, W, W_in, u, beta, dt):
    T = u.shape[0]
    K = W.shape[0]
    N = W.shape[1]
    x = np.zeros((T, N))
    x[0] = x0
    for t in range(1, T):
        
        x[t] = x[t-1]  + dt * ( np.tanh(beta * W @ x[t-1])  - x[t-1] + W_in @ u[t-1])
    return x

In [ ]:
def get_sparse_random_matrix(n_rows, n_cols, sparsity):
    M = np.zeros((n_rows, n_cols))
    for i in range(n_rows):
        sparse_indices = np.random.choice(n_cols, size=max(1, int(sparsity * n_cols)), replace=False)
        M[i,sparse_indices] = np.random.randn(len(sparse_indices))
        M[i] /= np.linalg.norm(M[i])
    return M

In [ ]:
reservoir_size = 1000
patterns_size = 500
use_sparse = True
model_to_use = 'softmax'
perc_throw_away = 0.25
for beta in np.linspace(3, 5, 10):
    for sparsity_level in [0.1, 0.3, 0.5]:
        np.random.seed(42)
        if (model_to_use == 'softmax' or model_to_use == 'softmax2'):    
            if model_to_use == 'softmax':
                if(use_sparse):
                    W = get_sparse_random_matrix(patterns_size, reservoir_size, sparsity_level)
                else:
                    W = np.random.randn(patterns_size, reservoir_size)
                    W /= np.linalg.norm(W, axis=1, keepdims=True)
            elif model_to_use == 'softmax2':
                if(use_sparse):
                    W_inner = get_sparse_random_matrix(patterns_size, reservoir_size, sparsity_level)
                    W_outer = get_sparse_random_matrix(patterns_size, reservoir_size, sparsity_level)
                else:
                    W_inner = np.random.randn(patterns_size, reservoir_size)
                    W_inner /= np.linalg.norm(W_inner, axis=1, keepdims=True)
                    W_outer = np.random.randn(patterns_size, reservoir_size)
                    W_outer /= np.linalg.norm(W_outer, axis=1, keepdims=True)
            W_in = np.random.randn(reservoir_size, 3)
            #W_in -= np.mean(W_in, keepdims=True, axis=1)
            W_in /= np.linalg.norm(W_in, axis=1, keepdims=True)

            x0 = np.random.randn(reservoir_size)
            x0 /= np.linalg.norm(x0)
            x0 = np.zeros(reservoir_size)
            if model_to_use == 'softmax':
                x = esn_softmax(x0, W, W_in, u_whitened, beta=beta, dt=dt)
            elif model_to_use == 'softmax2':
                x = esn_softmax2(x0, W_inner, W_outer, W_in, u_whitened, beta=beta, dt=dt)
        elif model_to_use == 'tanh':
            reservoir_size = 1000
            patterns_size = 1000
            W = np.random.randn(patterns_size, reservoir_size)
            W /= np.linalg.norm(W, axis=1, keepdims=True)
            W_in = np.random.randn(reservoir_size, 3)
            W_in /= np.linalg.norm(W_in, axis=1, keepdims=True)

            x0 = np.random.randn(reservoir_size)
            x0 /= np.linalg.norm(x0)
            x0 = np.zeros(reservoir_size)
            x = esn_tanh(x0, W, W_in, u_whitened, beta=beta, dt=dt)

        steps_to_throw_away = int(perc_throw_away * u_whitened.shape[0])
        from sklearn.linear_model import Ridge
        model = Ridge(alpha=1e-6, fit_intercept=False)
        model.fit(x, u_whitened)
        u_whitened_pred = model.predict(x)
        print(f"Beta: {beta:.4f}, Sparsity: {sparsity_level:.2f}, MSE: {np.mean((u_whitened - u_whitened_pred)**2):.6f}")
        %matplotlib inline
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        for i in range(3):
            ax[i].plot(u_whitened[steps_to_throw_away:, i], label='Original')
            ax[i].plot(u_whitened_pred[steps_to_throw_away:, i], label='Reconstructed', alpha=0.7)
            ax[i].legend()
        plt.show()

        %matplotlib inline
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(u_whitened[steps_to_throw_away:, 0], u_whitened[steps_to_throw_away:, 1], u_whitened[steps_to_throw_away:, 2], label='Original')
        ax.plot(u_whitened_pred[steps_to_throw_away:, 0], u_whitened_pred[steps_to_throw_away:, 1], u_whitened_pred[steps_to_throw_away:, 2], label='Reconstructed')
        ax.legend()
        plt.show()

In [ ]:
# we wanna do linear regression from x to u_whitened
from sklearn.linear_model import Ridge
model = Ridge(alpha=1e-6, fit_intercept=False)
model.fit(x, u_whitened)
u_whitened_pred = model.predict(x)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    ax[i].plot(u_whitened[:, i], label='Original')
    ax[i].plot(u_whitened_pred[:, i], label='Reconstructed', alpha=0.7)
    ax[i].legend()
plt.show()

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(u_whitened[:, 0], u_whitened[:, 1], u_whitened[:, 2], label='Original')
ax.plot(u_whitened_pred[:, 0], u_whitened_pred[:, 1], u_whitened_pred[:, 2], label='Reconstructed')
ax.legend()
plt.show()

In [ ]:
%matplotlib inline
fig, axs = plt.subplots(1, 3, figsize=(12, 5))
axs[0].plot(u_whitened[:, 0], u_whitened[:, 1], label='Original')
axs[0].plot(u_whitened_pred[:, 0], u_whitened_pred[:, 1], label='Reconstructed')
axs[0].set_xlabel('x')
axs[0].set_ylabel('y')
axs[0].legend()
axs[1].plot(u_whitened[:, 0], u_whitened[:, 2], label='Original')
axs[1].plot(u_whitened_pred[:, 0], u_whitened_pred[:, 2], label='Reconstructed')
axs[1].set_xlabel('x')
axs[1].set_ylabel('z')
axs[1].legend()
axs[2].plot(u_whitened[:, 1], u_whitened[:, 2], label='Original')
axs[2].plot(u_whitened_pred[:, 1], u_whitened_pred[:, 2], label='Reconstructed')
axs[2].set_xlabel('y')
axs[2].set_ylabel('z')
axs[2].legend()
plt.show()